# 1주차 8교시 · 처음 보는 신규회원 100명을 같은 기준으로 처리해 저장하기

**학생용 복습완성본**

오늘의 질문: **1~7교시에서 만든 절차를 처음 보는 파일에도 다시 적용할 수 있을까요?**

오늘의 완성 결과: 신규회원 100행을 67개 모델 입력 열로 바꾸고 CSV로 저장합니다.

학생이 막히면 정답 전체를 바로 보여주지 말고, [자료구조] → [입력] → [한 줄씩 읽기] → [정상 결과] 순으로 힌트를 줍니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def find_week1_data_dir():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in [
            base / "course_materials" / "matchmaking_data" / "01_week1",
            base / "matchmaking_data" / "01_week1",
            base / "01_week1",
        ]:
            if (candidate / "matchmaking_members_new.csv").exists():
                return candidate
    raise FileNotFoundError("01_week1 데이터 폴더를 찾지 못했습니다.")

DATA_DIR = find_week1_data_dir()
OUTPUT_DIR = DATA_DIR.parents[1] / "week1" / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("데이터 폴더:", DATA_DIR)


## Q1. 어떤 파일을 처리할지 먼저 확인합니다

**먼저 예상하기**

- exists()가 False이면 Pandas 코드부터 고쳐야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 파일을 읽기 전에 경로와 파일 존재 여부를 확인해, 코드 문제와 경로 문제를 구분합니다.

**현재 자료구조** DATA_DIR은 Path 객체이고 new_path도 파일 위치를 나타내는 Path 객체입니다.

**코드 읽기 도움** 슬래시(`/`)는 Path 객체에서 폴더와 파일명을 이어 하나의 파일 주소를 만드는 기호입니다.

**정상 결과** matchmaking_members_new.csv와 True

**오류가 나면** False가 나오거나 FileNotFoundError가 발생함. DATA_DIR을 출력하고 01_week1 폴더 안의 실제 파일명을 확인합니다.


In [ ]:
new_path = DATA_DIR / "matchmaking_members_new.csv"
print(new_path.name)
print(new_path.exists())

### 강사 설명 메모

- 학생에게 물을 질문: exists()가 False이면 Pandas 코드부터 고쳐야 할까요?
- 기대 답: 아니요. 먼저 폴더와 파일명을 확인합니다.
- 그대로 말할 문장: “슬래시(`/`)는 Path 객체에서 폴더와 파일명을 이어 하나의 파일 주소를 만드는 기호입니다.”
- 결과 확인: matchmaking_members_new.csv와 True
- 복구 순서: DATA_DIR을 출력하고 01_week1 폴더 안의 실제 파일명을 확인합니다.


## Q2. 신규회원 표의 크기와 상태를 확인합니다

**먼저 예상하기**

- 행이 100이라는 것은 무엇이 100개라는 뜻일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 정제 전에 행 수·열 수·중복·결측을 확인해 입력이 기대와 같은지 판단합니다.

**현재 자료구조** new_members는 100행×21열 DataFrame입니다. 한 행은 신규회원 한 명입니다.

**코드 읽기 도움** shape는 (행 수, 열 수) tuple을 돌려줍니다. duplicated는 같은 member_id가 앞에 있었는지 True/False로 표시합니다.

**정상 결과** (100, 21), 중복 0, 결측 0

**오류가 나면** shape가 다르거나 중복·결측이 0이 아님. 다른 CSV를 읽지 않았는지 new_path.name과 columns를 다시 확인합니다.


In [ ]:
new_members = pd.read_csv(new_path)
print(new_members.shape)
print("중복:", new_members.duplicated("member_id").sum())
print("결측:", new_members.isna().sum().sum())

### 강사 설명 메모

- 학생에게 물을 질문: 행이 100이라는 것은 무엇이 100개라는 뜻일까요?
- 기대 답: 신규회원 기록이 100개라는 뜻입니다.
- 그대로 말할 문장: “shape는 (행 수, 열 수) tuple을 돌려줍니다. duplicated는 같은 member_id가 앞에 있었는지 True/False로 표시합니다.”
- 결과 확인: (100, 21), 중복 0, 결측 0
- 복구 순서: 다른 CSV를 읽지 않았는지 new_path.name과 columns를 다시 확인합니다.


## Q3. 기존 1,000명에서 전처리 기준을 다시 만듭니다

**먼저 예상하기**

- 왜 신규회원 100명에서 중앙값을 배우면 안 될까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 새 회원에게 기준을 새로 배우지 않고, 기존 회원의 train 데이터에서만 중앙값·평균·범주 목록을 배우기 위해서입니다.

**현재 자료구조** members와 preferences는 DataFrame, X_train은 800행×14열 DataFrame, preprocessor는 처리 순서를 기억하는 ColumnTransformer입니다.

**코드 읽기 도움** 여기서 joined는 기존 회원표와 희망조건표를 member_id로 가로 결합한 새 DataFrame입니다.

**정상 결과** (1000, 37)

**오류가 나면** merge validate 오류 또는 연소득이 object로 남음. 중복 제거를 먼저 했는지, to_numeric 결과 dtype이 숫자인지 확인합니다.


In [ ]:
members = pd.read_csv(DATA_DIR / "matchmaking_members.csv")
members = members.drop_duplicates("member_id", keep="first").copy()
members["residence_region"] = members["residence_region"].str.strip().replace({"Seoul":"서울", "seoul":"서울"})
members["annual_income_10k_krw"] = pd.to_numeric(members["annual_income_10k_krw"], errors="coerce")
preferences = pd.read_csv(DATA_DIR / "matchmaking_preferences.csv")
joined = members.merge(preferences, on="member_id", how="left", validate="one_to_one")
print(joined.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 왜 신규회원 100명에서 중앙값을 배우면 안 될까요?
- 기대 답: 운영 중 들어온 새 데이터에 따라 기준이 계속 달라지기 때문입니다.
- 그대로 말할 문장: “여기서 joined는 기존 회원표와 희망조건표를 member_id로 가로 결합한 새 DataFrame입니다.”
- 결과 확인: (1000, 37)
- 복구 순서: 중복 제거를 먼저 했는지, to_numeric 결과 dtype이 숫자인지 확인합니다.


## Q4. 신규회원의 열을 기존 전처리 입력과 똑같이 맞춥니다

**먼저 예상하기**

- 새 열 네 개는 새 회원의 실제 응답이라고 볼 수 있을까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 저장된 처리기는 학습할 때 보았던 열 이름과 순서를 기대하므로 입력 구조를 맞춥니다.

**현재 자료구조** new_X는 100행×14열 DataFrame입니다. 원본 new_members는 그대로 남습니다.

**코드 읽기 도움** median()은 train의 가운데 값을, mode().iloc[0]은 train에서 가장 자주 나온 첫 번째 값을 가져옵니다.

**정상 결과** (100, 14)

**오류가 나면** NameError: X_train 또는 KeyError. Q3과 이 셀의 앞부분을 순서대로 실행하고 열 목록의 철자를 확인합니다.


In [ ]:
numeric_features = ["age","annual_income_10k_krw","profile_completion_rate","recommendation_response_rate_prior","consultation_count_prior","pref_income_min_10k_krw","preference_strictness"]
categorical_features = ["gender","residence_region","occupation_type","membership_tier","signup_channel","pref_education_min","deal_breakers"]
X = joined[numeric_features + categorical_features]
y = joined["meeting_scheduled"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
new_members["pref_income_min_10k_krw"] = np.nan
new_members["preference_strictness"] = X_train["preference_strictness"].median()
new_members["pref_education_min"] = X_train["pref_education_min"].mode().iloc[0]
new_members["deal_breakers"] = np.nan
new_X = new_members[numeric_features + categorical_features]
print(new_X.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 새 열 네 개는 새 회원의 실제 응답이라고 볼 수 있을까요?
- 기대 답: 아니요. 기존 회원의 희망조건 파일은 있지만, 신규회원 CSV에는 희망조건 열 네 개가 없습니다. 그래서 이번 실습에서는 일부를 `np.nan`으로 두고, 일부는 기존 train의 대표값으로 임시 보완합니다.
- 그대로 말할 문장: “median()은 train의 가운데 값을, mode().iloc[0]은 train에서 가장 자주 나온 첫 번째 값을 가져옵니다.”
- 결과 확인: (100, 14)
- 복구 순서: Q3과 이 셀의 앞부분을 순서대로 실행하고 열 목록의 철자를 확인합니다.


## Q5. 기준은 train에서 배우고 신규회원에는 적용만 합니다

**먼저 예상하기**

- 신규회원 줄에 fit_transform을 쓰면 어떤 원칙이 깨질까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 새 데이터가 들어와도 전처리 열 수와 순서를 같게 유지하는 실제 운영 원리를 확인합니다.

**현재 자료구조** preprocessor는 객체, new_processed는 100행×67열 NumPy 배열입니다.

**코드 읽기 도움** fit_transform은 기준을 배우며 적용하고, transform은 이미 배운 기준을 그대로 적용합니다.

**정상 결과** (800, 67)과 (100, 67)

**오류가 나면** 웨비나 범주에서 오류 또는 열 수 불일치. OneHotEncoder의 handle_unknown='ignore'와 new_X의 열 순서를 확인합니다.


In [ ]:
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("num", numeric_pipe, numeric_features), ("cat", categorical_pipe, categorical_features)])
train_processed = preprocessor.fit_transform(X_train)
new_processed = preprocessor.transform(new_X)
print(train_processed.shape, new_processed.shape)

### 강사 설명 메모

- 학생에게 물을 질문: 신규회원 줄에 fit_transform을 쓰면 어떤 원칙이 깨질까요?
- 기대 답: 신규 데이터에서 처리 기준을 다시 배우게 됩니다.
- 그대로 말할 문장: “fit_transform은 기준을 배우며 적용하고, transform은 이미 배운 기준을 그대로 적용합니다.”
- 결과 확인: (800, 67)과 (100, 67)
- 복구 순서: OneHotEncoder의 handle_unknown='ignore'와 new_X의 열 순서를 확인합니다.


## Q6. 처리 결과를 열 이름과 함께 CSV로 저장합니다

**먼저 예상하기**

- index=False를 빼면 파일에 무엇이 하나 더 생길까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 숫자 배열만 남기지 않고 회원 ID와 처리된 열 이름을 붙여 검토 가능한 파일로 보관합니다.

**현재 자료구조** processed_df는 100행×67열 DataFrame, export_df는 member_id까지 포함한 100행×68열 DataFrame입니다.

**코드 읽기 도움** assert는 반드시 맞아야 하는 조건을 검사합니다. 조건이 거짓이면 그 자리에서 멈춰 잘못된 결과 저장을 막습니다.

**정상 결과** week1_new_members_processed.csv 저장, (100, 68), 결측 0

**오류가 나면** PermissionError 또는 저장 파일에 Unnamed: 0 열이 생김. 엑셀에서 파일을 닫고 다시 저장하거나 index=False를 확인합니다.


In [ ]:
feature_names = preprocessor.get_feature_names_out()
processed_df = pd.DataFrame(new_processed, columns=feature_names)
export_df = pd.concat([new_members[["member_id"]].reset_index(drop=True), processed_df], axis=1)
output_path = OUTPUT_DIR / "week1_new_members_processed.csv"
export_df.to_csv(output_path, index=False)
assert export_df.shape == (100, 68)
assert export_df.isna().sum().sum() == 0
print(output_path, export_df.shape)

### 강사 설명 메모

- 학생에게 물을 질문: index=False를 빼면 파일에 무엇이 하나 더 생길까요?
- 기대 답: Pandas의 0~99 행 번호가 불필요한 열로 저장됩니다.
- 그대로 말할 문장: “assert는 반드시 맞아야 하는 조건을 검사합니다. 조건이 거짓이면 그 자리에서 멈춰 잘못된 결과 저장을 막습니다.”
- 결과 확인: week1_new_members_processed.csv 저장, (100, 68), 결측 0
- 복구 순서: 엑셀에서 파일을 닫고 다시 저장하거나 index=False를 확인합니다.


## 마무리 확인

- [ ] 오늘 결과가 **신규회원 100행을 67개 모델 입력 열로 바꾸고 CSV로 저장합니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
